# Agent 1 — Baseline Computation

Reads historical clean data from both pilot tables and computes
statistical baselines for each configured check type.

**Run this notebook before running the validation agent.**
**It must be re-run whenever new historical data is available.**

Reads from:
  - uc1_table_registry_b
  - uc1_column_registry_b
  - uc1_column_validation_registry_b
  - uc1_table_validation_registry_b
  - The actual pilot tables (raw transactions and inventory)

Writes to:
  - uc1_baseline_metrics_b

In [0]:
import uuid
from datetime import date, datetime, timedelta
from pyspark.sql import functions as F
from pyspark.sql import DataFrame
from pyspark.sql.types import (
    StructType, StructField,
    StringType, DateType, TimestampType,
    DoubleType, IntegerType, BooleanType
)
import logging

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

# ── Catalog and Schema ─────────────────────────────────────────────────────

dbutils.widgets.text("catalog", "learning_dev", "Target Catalog")
dbutils.widgets.text("schema", "s819534", "Target Schema")
CATALOG = dbutils.widgets.get("catalog")
SCHEMA  = dbutils.widgets.get("schema")

# ── Metadata Table References ──────────────────────────────────────────────

TABLE_REGISTRY              = f"{CATALOG}.{SCHEMA}.uc1_table_registry_b"
COLUMN_REGISTRY             = f"{CATALOG}.{SCHEMA}.uc1_column_registry_b"
COLUMN_VALIDATION_REGISTRY  = f"{CATALOG}.{SCHEMA}.uc1_column_validation_registry_b"
TABLE_VALIDATION_REGISTRY   = f"{CATALOG}.{SCHEMA}.uc1_table_validation_registry_b"
BASELINE_METRICS            = f"{CATALOG}.{SCHEMA}.uc1_baseline_metrics_b"

# ── Baseline Settings ──────────────────────────────────────────────────────

BASELINE_WINDOW_DAYS  = 30       # How many days of history to use
MIN_SAMPLE_COUNT      = 7        # Minimum days needed for a reliable baseline
BASELINE_DATE         = date.today()
COMPUTED_BY           = "05_baseline_computation"

# ── Baseline Schema ────────────────────────────────────────────────────────

BASELINE_SCHEMA = StructType([
    StructField("baseline_id",          StringType(),  False),
    StructField("fully_qualified_name", StringType(),  False),
    StructField("column_name",          StringType(),  True),
    StructField("check_type",           StringType(),  False),
    StructField("metric_name",          StringType(),  False),
    StructField("baseline_date",        DateType(),    False),
    StructField("baseline_window_days", IntegerType(), False),
    StructField("expected_value",       DoubleType(),  True),
    StructField("mean_value",           DoubleType(),  True),
    StructField("stddev_value",         DoubleType(),  True),
    StructField("min_value",            DoubleType(),  True),
    StructField("max_value",            DoubleType(),  True),
    StructField("p25_value",            DoubleType(),  True),
    StructField("p50_value",            DoubleType(),  True),
    StructField("p75_value",            DoubleType(),  True),
    StructField("p95_value",            DoubleType(),  True),
    StructField("sample_count",         IntegerType(), False),
    StructField("is_sufficient_data",   BooleanType(), False),
    StructField("baseline_status",      StringType(),  False),
    StructField("computed_timestamp",   TimestampType(), False),
    StructField("computed_by",          StringType(),  False),
])

print("=" * 60)
print("AGENT 1 — BASELINE COMPUTATION")
print("=" * 60)
print(f"Catalog             : {CATALOG}")
print(f"Schema              : {SCHEMA}")
print(f"Baseline date       : {BASELINE_DATE}")
print(f"Window days         : {BASELINE_WINDOW_DAYS}")
print(f"Min sample count    : {MIN_SAMPLE_COUNT}")
print(f"Computed by         : {COMPUTED_BY}")
print("=" * 60)

AGENT 1 — BASELINE COMPUTATION
Catalog             : learning_dev
Schema              : s819534
Baseline date       : 2026-11-11
Window days         : 30
Min sample count    : 7
Computed by         : 05_baseline_computation


## Step 1: Load Active Configuration from Metadata Tables

In [0]:
def load_active_tables() -> list:
    """
    Loads all active tables from the table registry.
    Returns a list of Row objects.
    """
    df = spark.table(TABLE_REGISTRY).filter(F.col("is_active") == True)
    tables = df.collect()
    print(f"Active tables found: {len(tables)}")
    for t in tables:
        print(f"  → {t['fully_qualified_name']}")
    return tables


def load_active_columns(fqn: str) -> list:
    """
    Returns active column names for a given table.
    """
    rows = (
        spark.table(COLUMN_REGISTRY)
             .filter(
                 (F.col("fully_qualified_name") == fqn) &
                 (F.col("is_active") == True)
             )
             .collect()
    )
    return [r["column_name"] for r in rows]


def load_column_validations(fqn: str) -> dict:
    """
    Returns a dict: column_name → set of validation_names
    Only active validations.
    """
    rows = (
        spark.table(COLUMN_VALIDATION_REGISTRY)
             .filter(
                 (F.col("fully_qualified_name") == fqn) &
                 (F.col("is_active") == True)
             )
             .collect()
    )
    result = {}
    for r in rows:
        col = r["column_name"]
        val = r["validation_name"]
        if col not in result:
            result[col] = set()
        result[col].add(val)
    return result


def load_table_validations(fqn: str) -> set:
    """
    Returns a set of active table-level validation names.
    """
    rows = (
        spark.table(TABLE_VALIDATION_REGISTRY)
             .filter(
                 (F.col("fully_qualified_name") == fqn) &
                 (F.col("is_active") == True)
             )
             .collect()
    )
    return {r["validation_name"] for r in rows}


# Load all active tables
active_tables = load_active_tables()
print(f"\nConfiguration loaded successfully.")

Active tables found: 1
  → comms_media_prd.exposure_data_asset.bronze_barrows_campaign

Configuration loaded successfully.


## Step 2: Baseline Computation Helper Functions

In [0]:
def get_baseline_window_dates(table_row) -> tuple:
    """
    Determines the start and end dates for the baseline window.
    Uses the business_date_column to filter historical data.

    For our pilot:
      End date   = 7 days ago (leaves room for controlled batches)
      Start date = End date minus BASELINE_WINDOW_DAYS
    """
    end_date   = BASELINE_DATE - timedelta(days=7)
    start_date = end_date - timedelta(days=BASELINE_WINDOW_DAYS)
    return start_date, end_date


def build_baseline_record(
    fqn         : str,
    column_name : str,
    check_type  : str,
    metric_name : str,
    stats       : dict,
    window_days : int
) -> dict:
    """
    Builds a single baseline record dictionary
    ready for writing to uc1_baseline_metrics_b.
    """
    sample_count  = int(stats.get("sample_count") or 0)
    is_sufficient = sample_count >= MIN_SAMPLE_COUNT

    mean_val = stats.get("mean_value")

    return {
        "baseline_id"          : str(uuid.uuid4()),
        "fully_qualified_name" : fqn,
        "column_name"          : column_name,
        "check_type"           : check_type,
        "metric_name"          : metric_name,
        "baseline_date"        : BASELINE_DATE,
        "baseline_window_days" : window_days,
        "expected_value"       : mean_val,
        "mean_value"           : mean_val,
        "stddev_value"         : stats.get("stddev_value"),
        "min_value"            : stats.get("min_value"),
        "max_value"            : stats.get("max_value"),
        "p25_value"            : stats.get("p25_value"),
        "p50_value"            : stats.get("p50_value"),
        "p75_value"            : stats.get("p75_value"),
        "p95_value"            : stats.get("p95_value"),
        "sample_count"         : sample_count,
        "is_sufficient_data"   : is_sufficient,
        "baseline_status"      : "VALID" if is_sufficient else "INSUFFICIENT_DATA",
        "computed_timestamp"   : datetime.now(),
        "computed_by"          : COMPUTED_BY
    }


def safe_float(value) -> float:
    """Safely converts a value to float, returning None if not possible."""
    try:
        return float(value) if value is not None else None
    except (TypeError, ValueError):
        return None


print("Helper functions loaded.")

Helper functions loaded.


In [0]:
import re
from pyspark.sql.types import ArrayType, StructType

# Registry paths follow the Unity Catalog schema view: the element of an
# array is called `items`, e.g. stores.selected.items.id.
# Legacy paths with [] (stores.selected[].id) are still accepted.
ARRAY_ELEMENT = "items"


def normalize_nested_path(path: str) -> str:
    p = (path or "").replace("[]", f".{ARRAY_ELEMENT}")
    while ".." in p:
        p = p.replace("..", ".")
    return p.strip(".")


def is_nested_path(column_name: str) -> bool:
    return "." in (column_name or "") or "[]" in (column_name or "")


def resolve_nested_column(df, column_name: str):
    """
    Resolves registry column paths like touchpoint.id or stores.selected.items.id
    into a DataFrame plus a materialized alias column name.
    Arrays are exploded with explode_outer so nested array elements can be
    baselined like primitive columns.
    """
    if not column_name:
        return df, None
    if column_name in df.columns:
        return df, column_name

    # Schema-aware walk: `items` explodes only when the current type is an
    # array, so a struct field literally named `items` stays a field.
    segs = normalize_nested_path(column_name).split(".")
    root = segs[0]
    if root not in df.columns:
        return df, None
    cur = F.col(f"`{root}`")
    dt  = df.schema[root].dataType
    explode_idx = 0
    for seg in segs[1:]:
        if isinstance(dt, ArrayType) and seg == ARRAY_ELEMENT:
            explode_idx += 1
            tmp = f"__ds_exploded_{explode_idx}"
            df  = df.withColumn(tmp, F.explode_outer(cur))
            cur = F.col(tmp)
            dt  = dt.elementType
        elif isinstance(dt, StructType) and seg in dt.fieldNames():
            cur = cur.getField(seg)
            dt  = dt[seg].dataType
        else:
            print(f"      Field '{seg}' not found while resolving {column_name}.")
            return df, None

    alias = "__ds_" + re.sub(r"[^0-9A-Za-z_]", "_", normalize_nested_path(column_name))
    df = df.withColumn(alias, cur)
    return df, alias


print("Nested column helpers ready.")

Nested column helpers ready.


## Step 3: Volume Anomaly Baseline

Computes daily row count statistics over the baseline window.
One baseline record per table.

In [0]:
def compute_volume_baseline(
    fqn                 : str,
    business_date_col   : str,
    start_date          : date,
    end_date            : date,
    window_days         : int
) -> dict:
    """
    Computes row count statistics per business date
    over the baseline window.

    Returns a stats dict or None on failure.
    """
    print(f"    [VOLUME_ANOMALY] Computing for {fqn}")
    print(f"      Window: {start_date} to {end_date}")

    try:
        df = spark.table(fqn)

        if business_date_col and business_date_col in df.columns:
            daily_counts = (
                df.filter(
                    (F.col(business_date_col) >= F.lit(str(start_date))) &
                    (F.col(business_date_col) <= F.lit(str(end_date)))
                )
                .groupBy(business_date_col)
                .agg(F.count("*").alias("row_count"))
            )
        else:
            # No business date column: use total row count
            total = df.count()
            print(f"      No business date column. Using total row count: {total}")
            return {
                "mean_value"   : float(total),
                "stddev_value" : 0.0,
                "min_value"    : float(total),
                "max_value"    : float(total),
                "p25_value"    : float(total),
                "p50_value"    : float(total),
                "p75_value"    : float(total),
                "p95_value"    : float(total),
                "sample_count" : 1
            }

        stats_row = daily_counts.agg(
            F.mean("row_count")
             .alias("mean_value"),
            F.stddev("row_count")
             .alias("stddev_value"),
            F.min("row_count")
             .alias("min_value"),
            F.max("row_count")
             .alias("max_value"),
            F.percentile_approx("row_count", 0.25)
             .alias("p25_value"),
            F.percentile_approx("row_count", 0.50)
             .alias("p50_value"),
            F.percentile_approx("row_count", 0.75)
             .alias("p75_value"),
            F.percentile_approx("row_count", 0.95)
             .alias("p95_value"),
            F.count("*")
             .alias("sample_count")
        ).collect()[0]

        result = {
            "mean_value"   : safe_float(stats_row["mean_value"]),
            "stddev_value" : safe_float(stats_row["stddev_value"]),
            "min_value"    : safe_float(stats_row["min_value"]),
            "max_value"    : safe_float(stats_row["max_value"]),
            "p25_value"    : safe_float(stats_row["p25_value"]),
            "p50_value"    : safe_float(stats_row["p50_value"]),
            "p75_value"    : safe_float(stats_row["p75_value"]),
            "p95_value"    : safe_float(stats_row["p95_value"]),
            "sample_count" : int(stats_row["sample_count"] or 0)
        }

        print(f"      Mean daily rows : {result['mean_value']:.1f}")
        print(f"      StdDev          : {result['stddev_value']:.1f}")
        print(f"      Min / Max       : {result['min_value']:.0f} / {result['max_value']:.0f}")
        print(f"      Sample days     : {result['sample_count']}")

        return result

    except Exception as e:
        print(f"      ERROR: {str(e)}")
        return None


print("Volume baseline function loaded.")

Volume baseline function loaded.


## Step 4: Null Spike Baseline

Computes daily null rate statistics for each monitored column.
One baseline record per column per table.

In [0]:
def compute_null_rate_baseline(
    fqn               : str,
    column_name       : str,
    business_date_col : str,
    start_date        : date,
    end_date          : date,
    window_days       : int
) -> dict:
    """
    Computes daily null rate statistics for a specific column.

    For columns marked with invert=true in custom_parameters
    (like _rescued_data), we compute the non-null rate instead.
    """
    print(f"    [NULL_SPIKE] Computing for {fqn} → {column_name}")

    try:
        df = spark.table(fqn)
        df, value_col = resolve_nested_column(df, column_name)

        if not value_col:
            print(f"      Column {column_name} not found. Skipping.")
            return None

        if business_date_col and business_date_col in df.columns:
            df_filtered = df.filter(
                (F.col(business_date_col) >= F.lit(str(start_date))) &
                (F.col(business_date_col) <= F.lit(str(end_date)))
            )

            daily_null_rates = (
                df_filtered
                .groupBy(business_date_col)
                .agg(
                    (
                        F.sum(F.col(value_col).isNull().cast("int")) /
                        F.count("*")
                    ).alias("null_rate"),
                    F.count("*").alias("total_records")
                )
            )
        else:
            total      = df.count()
            null_count = df.filter(F.col(value_col).isNull()).count()
            null_rate  = null_count / total if total > 0 else 0.0
            return {
                "mean_value"   : null_rate,
                "stddev_value" : 0.0,
                "min_value"    : null_rate,
                "max_value"    : null_rate,
                "p25_value"    : null_rate,
                "p50_value"    : null_rate,
                "p75_value"    : null_rate,
                "p95_value"    : null_rate,
                "sample_count" : 1
            }

        stats_row = daily_null_rates.agg(
            F.mean("null_rate")
             .alias("mean_value"),
            F.stddev("null_rate")
             .alias("stddev_value"),
            F.min("null_rate")
             .alias("min_value"),
            F.max("null_rate")
             .alias("max_value"),
            F.percentile_approx("null_rate", 0.25)
             .alias("p25_value"),
            F.percentile_approx("null_rate", 0.50)
             .alias("p50_value"),
            F.percentile_approx("null_rate", 0.75)
             .alias("p75_value"),
            F.percentile_approx("null_rate", 0.95)
             .alias("p95_value"),
            F.count("*")
             .alias("sample_count")
        ).collect()[0]

        result = {
            "mean_value"   : safe_float(stats_row["mean_value"]),
            "stddev_value" : safe_float(stats_row["stddev_value"]),
            "min_value"    : safe_float(stats_row["min_value"]),
            "max_value"    : safe_float(stats_row["max_value"]),
            "p25_value"    : safe_float(stats_row["p25_value"]),
            "p50_value"    : safe_float(stats_row["p50_value"]),
            "p75_value"    : safe_float(stats_row["p75_value"]),
            "p95_value"    : safe_float(stats_row["p95_value"]),
            "sample_count" : int(stats_row["sample_count"] or 0)
        }

        print(f"      Mean null rate  : {(result['mean_value'] or 0)*100:.4f}%")
        print(f"      Max null rate   : {(result['max_value'] or 0)*100:.4f}%")
        print(f"      Sample days     : {result['sample_count']}")

        return result

    except Exception as e:
        print(f"      ERROR: {str(e)}")
        return None


print("Null spike baseline function loaded.")

Null spike baseline function loaded.


## Step 5: Categorical Domain Baseline

Captures the set of distinct values seen in clean historical data.
Used to detect new or invalid values in subsequent loads.

In [0]:
def compute_categorical_baseline(
    fqn               : str,
    column_name       : str,
    business_date_col : str,
    start_date        : date,
    end_date          : date,
    window_days       : int
) -> dict:
    """
    Computes the distinct value count and value distribution
    for a categorical column over the baseline window.

    The distinct count serves as the baseline for CARDINALITY_DRIFT.
    For CATEGORICAL_DOMAIN, the valid value list comes from
    custom_parameters in the validation registry, not from here.
    """
    print(f"    [CATEGORICAL/CARDINALITY] Computing for {fqn} → {column_name}")

    try:
        df = spark.table(fqn)
        df, value_col = resolve_nested_column(df, column_name)

        if not value_col:
            print(f"      Column {column_name} not found. Skipping.")
            return None

        if business_date_col and business_date_col in df.columns:
            df_filtered = df.filter(
                (F.col(business_date_col) >= F.lit(str(start_date))) &
                (F.col(business_date_col) <= F.lit(str(end_date)))
            )
        else:
            df_filtered = df

        # Distinct count per day
        if business_date_col and business_date_col in df.columns:
            daily_cardinality = (
                df_filtered
                .groupBy(business_date_col)
                .agg(
                    F.countDistinct(value_col).alias("distinct_count")
                )
            )

            stats_row = daily_cardinality.agg(
                F.mean("distinct_count")
                 .alias("mean_value"),
                F.stddev("distinct_count")
                 .alias("stddev_value"),
                F.min("distinct_count")
                 .alias("min_value"),
                F.max("distinct_count")
                 .alias("max_value"),
                F.percentile_approx("distinct_count", 0.25)
                 .alias("p25_value"),
                F.percentile_approx("distinct_count", 0.50)
                 .alias("p50_value"),
                F.percentile_approx("distinct_count", 0.75)
                 .alias("p75_value"),
                F.percentile_approx("distinct_count", 0.95)
                 .alias("p95_value"),
                F.count("*")
                 .alias("sample_count")
            ).collect()[0]
        else:
            distinct = df_filtered.select(value_col).distinct().count()
            stats_row = {
                "mean_value"   : float(distinct),
                "stddev_value" : 0.0,
                "min_value"    : float(distinct),
                "max_value"    : float(distinct),
                "p25_value"    : float(distinct),
                "p50_value"    : float(distinct),
                "p75_value"    : float(distinct),
                "p95_value"    : float(distinct),
                "sample_count" : 1
            }

        result = {
            "mean_value"   : safe_float(stats_row["mean_value"]),
            "stddev_value" : safe_float(stats_row["stddev_value"]),
            "min_value"    : safe_float(stats_row["min_value"]),
            "max_value"    : safe_float(stats_row["max_value"]),
            "p25_value"    : safe_float(stats_row["p25_value"]),
            "p50_value"    : safe_float(stats_row["p50_value"]),
            "p75_value"    : safe_float(stats_row["p75_value"]),
            "p95_value"    : safe_float(stats_row["p95_value"]),
            "sample_count" : int(stats_row["sample_count"] or 0)
        }

        print(f"      Mean distinct   : {result['mean_value']:.1f}")
        print(f"      Min / Max       : {result['min_value']:.0f} / {result['max_value']:.0f}")
        print(f"      Sample days     : {result['sample_count']}")

        return result

    except Exception as e:
        print(f"      ERROR: {str(e)}")
        return None


print("Categorical baseline function loaded.")

Categorical baseline function loaded.


## Step 6: Numeric Distribution Baseline

Computes value distribution statistics for numeric columns.
Used for NUMERIC_RANGE and DISTRIBUTION_DRIFT checks.

In [0]:
def compute_numeric_baseline(
    fqn               : str,
    column_name       : str,
    business_date_col : str,
    start_date        : date,
    end_date          : date,
    window_days       : int
) -> dict:
    """
    Computes value distribution statistics for a numeric column.
    Operates on all non-null values within the baseline window.
    """
    print(f"    [NUMERIC_RANGE] Computing for {fqn} → {column_name}")

    try:
        df = spark.table(fqn)
        df, value_col = resolve_nested_column(df, column_name)

        if not value_col:
            print(f"      Column {column_name} not found. Skipping.")
            return None

        if business_date_col and business_date_col in df.columns:
            df_filtered = df.filter(
                (F.col(business_date_col) >= F.lit(str(start_date))) &
                (F.col(business_date_col) <= F.lit(str(end_date)))
            )
        else:
            df_filtered = df

        # Only non-null values
        df_col = (
            df_filtered
            .filter(F.col(value_col).isNotNull())
            .select(F.col(value_col).alias("value_col"))
        )

        total_records = df_col.count()
        if total_records == 0:
            print(f"      No non-null records found for {column_name}. Skipping.")
            return None

        stats_row = df_col.agg(
            F.mean("value_col")
             .alias("mean_value"),
            F.stddev("value_col")
             .alias("stddev_value"),
            F.min("value_col")
             .alias("min_value"),
            F.max("value_col")
             .alias("max_value"),
            F.percentile_approx("value_col", 0.25)
             .alias("p25_value"),
            F.percentile_approx("value_col", 0.50)
             .alias("p50_value"),
            F.percentile_approx("value_col", 0.75)
             .alias("p75_value"),
            F.percentile_approx("value_col", 0.95)
             .alias("p95_value"),
            F.count("*")
             .alias("sample_count")
        ).collect()[0]

        result = {
            "mean_value"   : safe_float(stats_row["mean_value"]),
            "stddev_value" : safe_float(stats_row["stddev_value"]),
            "min_value"    : safe_float(stats_row["min_value"]),
            "max_value"    : safe_float(stats_row["max_value"]),
            "p25_value"    : safe_float(stats_row["p25_value"]),
            "p50_value"    : safe_float(stats_row["p50_value"]),
            "p75_value"    : safe_float(stats_row["p75_value"]),
            "p95_value"    : safe_float(stats_row["p95_value"]),
            "sample_count" : int(stats_row["sample_count"] or 0)
        }

        print(f"      Mean value      : {result['mean_value']:.4f}")
        print(f"      StdDev          : {result['stddev_value']:.4f}")
        print(f"      Min / Max       : {result['min_value']:.4f} / {result['max_value']:.4f}")
        print(f"      Sample records  : {result['sample_count']}")

        return result

    except Exception as e:
        print(f"      ERROR: {str(e)}")
        return None


print("Numeric baseline function loaded.")

Numeric baseline function loaded.


## Step 7: Write Baseline Records to uc1_baseline_metrics_b

In [0]:
def write_baseline_records(records: list) -> int:
    """
    Writes baseline records to uc1_baseline_metrics_b.
    Uses MERGE to handle reruns gracefully.
    Existing baselines for the same table/column/check/date
    are updated rather than duplicated.
    """
    if not records:
        print("  No records to write.")
        return 0

    # Convert to tuples in schema column order
    column_order = [field.name for field in BASELINE_SCHEMA.fields]

    rows = [
        tuple(r.get(col) for col in column_order)
        for r in records
    ]

    df = spark.createDataFrame(rows, schema=BASELINE_SCHEMA)

    df.createOrReplaceTempView("baseline_staging")

    spark.sql(f"""
        MERGE INTO {BASELINE_METRICS} AS target
        USING baseline_staging AS source
        ON  target.fully_qualified_name = source.fully_qualified_name
        AND target.column_name          = source.column_name
        AND target.check_type           = source.check_type
        AND target.metric_name          = source.metric_name
        AND target.baseline_date        = source.baseline_date
        WHEN MATCHED THEN
            UPDATE SET *
        WHEN NOT MATCHED THEN
            INSERT *
    """)

    print(f"  Written {len(records)} baseline records.")
    return len(records)


print("Write function loaded.")

Write function loaded.


## Step 8: Run Baseline Computation for All Active Tables

In [0]:
all_baseline_records = []
tables_processed     = 0
tables_skipped       = 0
tables_failed        = 0

for table_row in active_tables:
    fqn               = table_row["fully_qualified_name"]
    biz_date_col      = table_row["business_date_column"] or ""

    print(f"\n{'='*60}")
    print(f"Processing: {fqn}")
    print(f"Business date column: {biz_date_col or 'None'}")
    print(f"{'='*60}")

    try:
        # ── Load configuration for this table ─────────────────────────────
        active_columns    = load_active_columns(fqn)
        col_validations   = load_column_validations(fqn)
        table_validations = load_table_validations(fqn)

        print(f"Active columns     : {len(active_columns)}")
        print(f"Table validations  : {table_validations}")
        print(f"Column validations : {len(col_validations)} columns configured")

        if not table_validations and not col_validations:
            print("No validations configured. Skipping.")
            tables_skipped += 1
            continue

        # ── Compute baseline window dates ──────────────────────────────────
        start_date, end_date = get_baseline_window_dates(table_row)
        print(f"Baseline window    : {start_date} to {end_date}")
        print()

        # ── Table-Level Baselines ──────────────────────────────────────────

        # VOLUME_ANOMALY
        if "VOLUME_ANOMALY" in table_validations:
            stats = compute_volume_baseline(
                fqn, biz_date_col, start_date, end_date, BASELINE_WINDOW_DAYS
            )
            if stats:
                all_baseline_records.append(
                    build_baseline_record(
                        fqn         = fqn,
                        column_name = "__table__",
                        check_type  = "VOLUME_ANOMALY",
                        metric_name = "daily_row_count",
                        stats       = stats,
                        window_days = BASELINE_WINDOW_DAYS
                    )
                )

        # SCHEMA_CONFORMANCE baseline = current column count
        if "SCHEMA_CONFORMANCE" in table_validations:
            try:
                schema_cols = spark.table(fqn).columns
                col_count   = len(schema_cols)
                stats = {
                    "mean_value"   : float(col_count),
                    "stddev_value" : 0.0,
                    "min_value"    : float(col_count),
                    "max_value"    : float(col_count),
                    "p25_value"    : float(col_count),
                    "p50_value"    : float(col_count),
                    "p75_value"    : float(col_count),
                    "p95_value"    : float(col_count),
                    "sample_count" : 1
                }
                all_baseline_records.append(
                    build_baseline_record(
                        fqn         = fqn,
                        column_name = "__table__",
                        check_type  = "SCHEMA_CONFORMANCE",
                        metric_name = "column_count",
                        stats       = stats,
                        window_days = BASELINE_WINDOW_DAYS
                    )
                )
                print(f"    [SCHEMA_CONFORMANCE] Column count baseline: {col_count}")
            except Exception as e:
                print(f"    [SCHEMA_CONFORMANCE] ERROR: {str(e)}")

        # MALFORMED_RECORDS baseline (for Autoloader tables)
        if "MALFORMED_RECORDS" in table_validations:
            stats = compute_null_rate_baseline(
                fqn, "_rescued_data", biz_date_col,
                start_date, end_date, BASELINE_WINDOW_DAYS
            )
            if stats:
                # For malformed records we track the non-null rate
                # In clean data this should be ~0%
                all_baseline_records.append(
                    build_baseline_record(
                        fqn         = fqn,
                        column_name = "__table__",
                        check_type  = "MALFORMED_RECORDS",
                        metric_name = "rescued_data_rate",
                        stats       = stats,
                        window_days = BASELINE_WINDOW_DAYS
                    )
                )

        # ── Column-Level Baselines ─────────────────────────────────────────

        for col_name in active_columns:
            checks_for_col = col_validations.get(col_name, set())

            if not checks_for_col:
                continue

            print(f"\n  Column: {col_name}")
            print(f"  Checks: {checks_for_col}")

            # NULL_SPIKE
            if "NULL_SPIKE" in checks_for_col:
                stats = compute_null_rate_baseline(
                    fqn, col_name, biz_date_col,
                    start_date, end_date, BASELINE_WINDOW_DAYS
                )
                if stats:
                    all_baseline_records.append(
                        build_baseline_record(
                            fqn         = fqn,
                            column_name = col_name,
                            check_type  = "NULL_SPIKE",
                            metric_name = "null_rate",
                            stats       = stats,
                            window_days = BASELINE_WINDOW_DAYS
                        )
                    )

            # NUMERIC_RANGE
            if "NUMERIC_RANGE" in checks_for_col:
                stats = compute_numeric_baseline(
                    fqn, col_name, biz_date_col,
                    start_date, end_date, BASELINE_WINDOW_DAYS
                )
                if stats:
                    all_baseline_records.append(
                        build_baseline_record(
                            fqn         = fqn,
                            column_name = col_name,
                            check_type  = "NUMERIC_RANGE",
                            metric_name = "value_distribution",
                            stats       = stats,
                            window_days = BASELINE_WINDOW_DAYS
                        )
                    )

            # CATEGORICAL_DOMAIN and CARDINALITY_DRIFT
            if "CATEGORICAL_DOMAIN" in checks_for_col or \
               "CARDINALITY_DRIFT"  in checks_for_col:
                stats = compute_categorical_baseline(
                    fqn, col_name, biz_date_col,
                    start_date, end_date, BASELINE_WINDOW_DAYS
                )
                if stats:
                    if "CATEGORICAL_DOMAIN" in checks_for_col:
                        all_baseline_records.append(
                            build_baseline_record(
                                fqn         = fqn,
                                column_name = col_name,
                                check_type  = "CATEGORICAL_DOMAIN",
                                metric_name = "distinct_value_count",
                                stats       = stats,
                                window_days = BASELINE_WINDOW_DAYS
                            )
                        )
                    if "CARDINALITY_DRIFT" in checks_for_col:
                        all_baseline_records.append(
                            build_baseline_record(
                                fqn         = fqn,
                                column_name = col_name,
                                check_type  = "CARDINALITY_DRIFT",
                                metric_name = "distinct_value_count",
                                stats       = stats,
                                window_days = BASELINE_WINDOW_DAYS
                            )
                        )

            # CROSS_COLUMN_CONSISTENCY baseline
            # The baseline for this check is a 0% violation rate
            # in clean data. We record this explicitly.
            if "CROSS_COLUMN_CONSISTENCY" in checks_for_col:
                stats = {
                    "mean_value"   : 0.0,
                    "stddev_value" : 0.0,
                    "min_value"    : 0.0,
                    "max_value"    : 0.0,
                    "p25_value"    : 0.0,
                    "p50_value"    : 0.0,
                    "p75_value"    : 0.0,
                    "p95_value"    : 0.0,
                    "sample_count" : BASELINE_WINDOW_DAYS
                }
                all_baseline_records.append(
                    build_baseline_record(
                        fqn         = fqn,
                        column_name = col_name,
                        check_type  = "CROSS_COLUMN_CONSISTENCY",
                        metric_name = "violation_rate",
                        stats       = stats,
                        window_days = BASELINE_WINDOW_DAYS
                    )
                )
                print(f"    [CROSS_COLUMN_CONSISTENCY] Baseline: 0% violation rate")

        tables_processed += 1
        print(f"\n  Completed. Running total: {len(all_baseline_records)} records.")

    except Exception as e:
        print(f"\n  FAILED: {str(e)}")
        tables_failed += 1

print(f"\n{'='*60}")
print("Loop complete.")
print(f"  Tables processed : {tables_processed}")
print(f"  Tables skipped   : {tables_skipped}")
print(f"  Tables failed    : {tables_failed}")
print(f"  Records to write : {len(all_baseline_records)}")
print(f"{'='*60}")


Processing: comms_media_prd.exposure_data_asset.bronze_barrows_campaign
Business date column: created
Active columns     : 24
Table validations  : {'MALFORMED_RECORDS', 'SCHEMA_DRIFT', 'DUPLICATE_CHECK', 'VOLUME_ANOMALY'}
Column validations : 23 columns configured
Baseline window    : 2026-10-05 to 2026-11-04

    [VOLUME_ANOMALY] Computing for comms_media_prd.exposure_data_asset.bronze_barrows_campaign
      Window: 2026-10-05 to 2026-11-04
      Mean daily rows : 4.4
      StdDev          : 4.2
      Min / Max       : 1 / 15
      Sample days     : 14
    [NULL_SPIKE] Computing for comms_media_prd.exposure_data_asset.bronze_barrows_campaign → _rescued_data
      Mean null rate  : 0.0000%
      Max null rate   : 0.0000%
      Sample days     : 14

  Column: id
  Checks: {'NULL_SPIKE', 'CARDINALITY_DRIFT'}
    [NULL_SPIKE] Computing for comms_media_prd.exposure_data_asset.bronze_barrows_campaign → id
      Mean null rate  : 0.0000%
      Max null rate   : 0.0000%
      Sample days    

## Step 9: Write All Baseline Records

In [0]:
print("Writing baseline records to uc1_baseline_metrics_b...")
records_written = write_baseline_records(all_baseline_records)
print(f"Write complete. {records_written} records written.")

Writing baseline records to uc1_baseline_metrics_b...
  Written 28 baseline records.
Write complete. 28 records written.


## Step 10: Verify Baseline Results

In [0]:
print("=" * 60)
print("BASELINE VERIFICATION")
print("=" * 60)

# Summary by table and check type
print("\n1. Baselines by table and check type:\n")
spark.sql(f"""
    SELECT
        fully_qualified_name,
        check_type,
        column_name,
        metric_name,
        ROUND(mean_value, 4)    AS mean_value,
        ROUND(stddev_value, 4)  AS stddev_value,
        ROUND(min_value, 4)     AS min_value,
        ROUND(max_value, 4)     AS max_value,
        sample_count,
        is_sufficient_data,
        baseline_status
    FROM {BASELINE_METRICS}
    WHERE baseline_date = '{BASELINE_DATE}'
    ORDER BY
        fully_qualified_name,
        check_type,
        column_name
""").show(100, truncate=False)

# Volume anomaly baselines specifically
print("\n2. Volume anomaly baselines:\n")
spark.sql(f"""
    SELECT
        fully_qualified_name,
        ROUND(mean_value, 1)    AS mean_daily_rows,
        ROUND(stddev_value, 1)  AS stddev,
        ROUND(min_value, 0)     AS min_rows,
        ROUND(max_value, 0)     AS max_rows,
        sample_count,
        baseline_status
    FROM {BASELINE_METRICS}
    WHERE baseline_date = '{BASELINE_DATE}'
      AND check_type = 'VOLUME_ANOMALY'
""").show(truncate=False)

# Null rate baselines
print("\n3. Null rate baselines (NULL_SPIKE):\n")
spark.sql(f"""
    SELECT
        fully_qualified_name,
        column_name,
        ROUND(mean_value * 100, 4)  AS mean_null_pct,
        ROUND(max_value * 100, 4)   AS max_null_pct,
        sample_count,
        baseline_status
    FROM {BASELINE_METRICS}
    WHERE baseline_date = '{BASELINE_DATE}'
      AND check_type = 'NULL_SPIKE'
    ORDER BY fully_qualified_name, column_name
""").show(truncate=False)

# Cardinality baselines
print("\n4. Cardinality baselines:\n")
spark.sql(f"""
    SELECT
        fully_qualified_name,
        column_name,
        check_type,
        ROUND(mean_value, 1)    AS mean_distinct_count,
        ROUND(min_value, 0)     AS min_distinct,
        ROUND(max_value, 0)     AS max_distinct,
        sample_count,
        baseline_status
    FROM {BASELINE_METRICS}
    WHERE baseline_date = '{BASELINE_DATE}'
      AND check_type IN ('CATEGORICAL_DOMAIN', 'CARDINALITY_DRIFT')
    ORDER BY fully_qualified_name, column_name
""").show(truncate=False)

# Overall summary
print("\n5. Overall baseline summary:\n")
spark.sql(f"""
    SELECT
        fully_qualified_name,
        COUNT(*)                                                AS total_baselines,
        SUM(CASE WHEN is_sufficient_data THEN 1 ELSE 0 END)    AS sufficient,
        SUM(CASE WHEN NOT is_sufficient_data THEN 1 ELSE 0 END) AS insufficient,
        COUNT(DISTINCT check_type)                             AS distinct_checks
    FROM {BASELINE_METRICS}
    WHERE baseline_date = '{BASELINE_DATE}'
    GROUP BY fully_qualified_name
    ORDER BY fully_qualified_name
""").show(truncate=False)

BASELINE VERIFICATION

1. Baselines by table and check type:

+-----------------------------------------------------------+------------------------+----------------------------+---------------------------------------+----------+------------+---------+---------+------------+------------------+---------------+
|fully_qualified_name                                       |check_type              |column_name                 |metric_name                            |mean_value|stddev_value|min_value|max_value|sample_count|is_sufficient_data|baseline_status|
+-----------------------------------------------------------+------------------------+----------------------------+---------------------------------------+----------+------------+---------+---------+------------+------------------+---------------+
|comms_media_prd.exposure_data_asset.bronze_barrows_campaign|CARDINALITY_DRIFT       |id                          |distinct_value_count                   |1.0       |0.0         |1.0      |1.0  

## Summary

In [0]:
print("\n" + "=" * 60)
print("BASELINE COMPUTATION COMPLETE")
print("=" * 60)
print(f"Tables processed         : {tables_processed}")
print(f"Tables skipped           : {tables_skipped}")
print(f"Tables failed            : {tables_failed}")
print(f"Baseline records written : {records_written}")
print(f"Baseline date            : {BASELINE_DATE}")
print(f"Baseline window          : {BASELINE_WINDOW_DAYS} days")
print(f"Completed at             : {datetime.now()}")
print("=" * 60)

if tables_failed > 0:
    print("\nWARNING: Some tables failed.")
    print("Check the output above for error details.")

if records_written == 0:
    print("\nACTION REQUIRED: No baselines were computed.")
    print("Ensure tables are registered and validations are configured.")
else:
    print("\nBaselines are ready.")
    print("You can now run the Agent 1 validation notebook.")


BASELINE COMPUTATION COMPLETE
Tables processed         : 1
Tables skipped           : 0
Tables failed            : 0
Baseline records written : 28
Baseline date            : 2026-11-11
Baseline window          : 30 days
Completed at             : 2026-11-11 16:21:37.872958

Baselines are ready.
You can now run the Agent 1 validation notebook.
